1.LOAD DATASET

In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

PLATFORM = "instagram"
SEEDS = [42, 123, 2024, 7, 99]
DATA_DIR = "../data_preprocess/processed_data/ml_data/"

results = []

for seed in SEEDS:
    train_df = pd.read_csv(f"{DATA_DIR}{PLATFORM}_train_seed{seed}.csv")
    test_df  = pd.read_csv(f"{DATA_DIR}{PLATFORM}_test_seed{seed}.csv")

    neg, pos = train_df["popularity"].value_counts()[0], train_df["popularity"].value_counts()[1]
    ratio = neg / pos

    X_train = train_df.drop(columns=["post_id", "user_id", "popularity"])
    y_train = train_df["popularity"]
    X_test  = test_df.drop(columns=["post_id", "user_id", "popularity"])
    y_test  = test_df["popularity"]

    model = XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=ratio, n_estimators=500, learning_rate=0.02,
        max_depth=6, min_child_weight=1, gamma=0.1,
        subsample=0.8, colsample_bytree=0.5,
        random_state=seed, n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "seed": seed,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })
    print(f"[seed {seed}] done | F1={results[-1]['f1']:.4f} | ROC-AUC={results[-1]['roc_auc']:.4f}")

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print(f"{PLATFORM.upper()} BASELINE — MEAN ± STD ACROSS {len(SEEDS)} SEEDS")
print("="*60)
print(results_df.set_index("seed"))
print("\nMean ± Std:")
summary = results_df.drop(columns="seed").agg(["mean", "std"])
print(summary)

[seed 42] done | F1=0.5894 | ROC-AUC=0.8679
[seed 123] done | F1=0.6064 | ROC-AUC=0.8756
[seed 2024] done | F1=0.6010 | ROC-AUC=0.8760
[seed 7] done | F1=0.6045 | ROC-AUC=0.8770
[seed 99] done | F1=0.6093 | ROC-AUC=0.8810

INSTAGRAM BASELINE — MEAN ± STD ACROSS 5 SEEDS
      accuracy  precision    recall        f1   roc_auc
seed                                                   
42    0.788707   0.482868  0.756400  0.589447  0.867920
123   0.794502   0.492276  0.789430  0.606407  0.875625
2024  0.791356   0.487417  0.783650  0.601013  0.875969
7     0.792681   0.489514  0.790256  0.604548  0.876994
99    0.795496   0.493846  0.795211  0.609301  0.880995

Mean ± Std:
      accuracy  precision    recall        f1   roc_auc
mean  0.792548   0.489184  0.782989  0.602143  0.875501
std   0.002679   0.004313  0.015420  0.007708  0.004747


In [2]:
results_df.insert(0, "model", "baseline_metadata")
results_df.insert(0, "platform", PLATFORM)

import os
os.makedirs("../results", exist_ok=True)
results_df.to_csv(f"../results/{PLATFORM}_baseline_metadata.csv", index=False)
print(f"Saved: ../results/{PLATFORM}_baseline_metadata.csv")

Saved: ../results/instagram_baseline_metadata.csv
